In [2]:
import os, glob, joblib, sklearn, numpy, pandas, scipy, json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (ManagedOnlineEndpoint, ManagedOnlineDeployment,
                                  Model, Environment, CodeConfiguration)
from azure.identity import DefaultAzureCredential

ml_client = MLClient(DefaultAzureCredential(), "9a94ea5e-07bd-42b7-b7d6-58b18d46274e",
                     "rg-project1-hasiba0045", "mlw-project1-hasiba0045")

# 0. Delete the earlier failed endpoints (stops any cost and frees CPU quota)
for name in ["knn-maint-hasiba0045", "knn-maint-hasiba0045b"]:
    try:
        ml_client.online_endpoints.begin_delete(name=name).result(); print("Deleted:", name)
    except Exception:
        print("Not found (ok):", name)

# 1. Retrain the best notebook model (K=1, uniform, p=1, same split) and save it with joblib
df = pd.read_csv("../data/ai4i_clean.csv")
NUM = ["air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min"]; CAT = ["type"]
df[NUM] = df[NUM].astype(float)
X_train, _, y_train, _ = train_test_split(df[NUM + CAT], df["machine_failure"],
                                          test_size=0.2, stratify=df["machine_failure"], random_state=42)
pipe = Pipeline([("prep", ColumnTransformer([("num", StandardScaler(), NUM),
                                             ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])),
                 ("knn", KNeighborsClassifier(n_neighbors=1, weights="uniform", p=1))]).fit(X_train, y_train)
os.makedirs("/tmp/knn_model", exist_ok=True)
joblib.dump(pipe, "/tmp/knn_model/model.pkl")

# 2. Write the scoring script and a sample request (both are saved in the repo)
os.makedirs("../deployment", exist_ok=True)
with open("../deployment/score.py", "w") as f:
    f.write('''import os, glob, json, joblib
import pandas as pd

def init():
    global model
    path = glob.glob(os.path.join(os.environ["AZUREML_MODEL_DIR"], "**", "*.pkl"), recursive=True)[0]
    model = joblib.load(path)

def run(raw_data):
    data = json.loads(raw_data)["input_data"]
    df = pd.DataFrame(data["data"], columns=data["columns"])
    return model.predict(df).tolist()
''')
sample = {"input_data": {
    "columns": ["air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min", "type"],
    "index": [0, 1],
    "data": [[298.1, 308.6, 1551.0, 42.8, 0.0, "M"],
             [302.5, 310.9, 1325.0, 68.2, 215.0, "L"]]}}
with open("../deployment/sample-request.json", "w") as f:
    json.dump(sample, f, indent=2)

# 3. Environment pinned to the exact package versions used to train the model
with open("/tmp/env.yml", "w") as f:
    f.write(f"""name: knn-env
channels: [conda-forge]
dependencies:
  - python=3.10
  - pip
  - pip:
    - azureml-inference-server-http
    - scikit-learn=={sklearn.__version__}
    - numpy=={numpy.__version__}
    - pandas=={pandas.__version__}
    - scipy=={scipy.__version__}
    - joblib=={joblib.__version__}
""")
env = Environment(name="knn-sklearn-env", conda_file="/tmp/env.yml",
                  image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest")
model = ml_client.models.create_or_update(Model(path="/tmp/knn_model/model.pkl",
                                                name="knn-maintenance-joblib", type="custom_model"))

# 4. Create the endpoint and the deployment (image build ~10 min + deployment ~5 min)
ENDPOINT_NAME = "knn-maint-hasiba0045c"
ml_client.online_endpoints.begin_create_or_update(
    ManagedOnlineEndpoint(name=ENDPOINT_NAME, auth_mode="key")).result()
print("Endpoint created:", ENDPOINT_NAME)
ml_client.online_deployments.begin_create_or_update(ManagedOnlineDeployment(
    name="blue", endpoint_name=ENDPOINT_NAME, model=model, environment=env,
    code_configuration=CodeConfiguration(code="../deployment", scoring_script="score.py"),
    instance_type="Standard_DS2_v2", instance_count=1)).result()
ep = ml_client.online_endpoints.get(ENDPOINT_NAME); ep.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(ep).result()
print("Deployment ready")

# 5. Test the endpoint with the sample request
response = ml_client.online_endpoints.invoke(endpoint_name=ENDPOINT_NAME, deployment_name="blue",
                                             request_file="../deployment/sample-request.json")
print("Prediction (0 = OK, 1 = failure):", response)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Uploading model.pkl (< 1 MB): 100%|██████████| 1.24M/1.24M [00:00<00:00, 17.4MB/s]


Uploading deployment (0.0 MBs): 100%|██████████| 802/802 [00:00<00:00, 6349.15it/s]




...Deleted: knn-maint-hasiba0045
..............................................................Deleted: knn-maint-hasiba0045b
Endpoint created: knn-maint-hasiba0045c
..............................................................................................................................................Deployment ready
Prediction (0 = OK, 1 = failure): [0, 1]


In [3]:
import requests
keys = ml_client.online_endpoints.get_keys(name=ENDPOINT_NAME)
url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
with open("../deployment/sample-request.json") as f:
    payload = json.load(f)
r = requests.post(url, json=payload, timeout=30,
                  headers={"Content-Type": "application/json", "Authorization": f"Bearer {keys.primary_key}"})
print("HTTP status:", r.status_code)
print("Prediction:", r.text)

HTTP status: 200
Prediction: [0, 1]


In [4]:
ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
print("Endpoint deleted")

........................................................Endpoint deleted
